In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import wandb
from datasets import Dataset
from transformers import AutoTokenizer
from finetune import train_eval_model
from experiment import preprocess_text

In [2]:
models_config = [
    ("sentence-transformers/all-MiniLM-L12-v2", "outputs/emotion-classifier-minilm"),
    ("sentence-transformers/LaBSE", "outputs/emotion-classifier-labse"),
    ("intfloat/multilingual-e5-large", "outputs/emotion-classifier-e5")
]
label_config = [
    {0: 'neutral', 0.5: 'present'}, 
    {0: 'neutral', 0.5: 'medium', 1.5: 'high'},
    {0: 'neutral', 0.5: 'moderate', 1.0: 'high', 1.5: 'strong', 2.0: 'extreme'}
]

In [3]:
def preprocess(example):
    return {'text': preprocess_text(example['text']), 'labels': example['labels']}

def load_emotionality_dataset(tokenizer, label_bounds = {0: 'present'}):
    DATA_PATH = Path("../data/2026-05-21/corpus_emotions.json")
    data = pd.read_json(DATA_PATH)
    labels = pd.cut(
        data['emotionality'], 
        bins=list(label_bounds.keys()) + [np.max(data['emotionality'])+1],
        labels=list(label_bounds.values()),
        right=False
    ).astype('object')
    data = Dataset.from_dict({'text': data['content'], 'labels': labels})
    data = data.map(preprocess)
    labels_map = {label: ind for ind, label in enumerate(set(data['labels']))}

    def tokenize(example):
        labels = list(map(lambda m: labels_map.get(m), example['labels']))
        example = tokenizer(example["text"], padding="max_length", truncation=True)
        example['labels'] = labels
        return example

    final_dst = data.map(tokenize, batched=True, remove_columns=['text'])
    final_dst.set_format(type="torch")
    return final_dst, labels_map

In [4]:
for model_name, output_dir in models_config:
    for ind, label_bounds in enumerate(label_config):
        print(f"Training model for emotion detection with model {model_name} using discretization to {len(label_bounds)} classes")
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        ds, labels_map = load_emotionality_dataset(tokenizer, label_bounds=label_bounds)
        run = wandb.init(project="emotion-detection-lt", name=f"{model_name}-{len(label_bounds)}class-{ind}")
        train_eval_model(
            model_name=model_name,
            dataset=ds,
            labels_map=labels_map,
            output_dir=f"{output_dir}-{ind}", 
            batch_size=16, 
            num_epochs=20, 
            eval_batch_size=16,
            report_to=['tensorboard', 'wandb']
        )

Training model for emotion detection with model sentence-transformers/all-MiniLM-L12-v2 using discretization to 2 classes


tokenizer_config.json:   0%|          | 0.00/352 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

loading configuration file config.json from cache at /home/paudan/.cache/huggingface/hub/models--intfloat--multilingual-e5-large/snapshots/3d7cfbdacd47fdda877c5cd8a79fbcc4f2a574f3/config.json
Model config XLMRobertaConfig {
  "architectures": [
    "XLMRobertaModel"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "dtype": "float32",
  "eos_token_id": 2,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 1024,
  "id2label": {
    "0": "moderate",
    "1": "strong",
    "2": "extreme",
    "3": "neutral",
    "4": "high"
  },
  "initializer_range": 0.02,
  "intermediate_size": 4096,
  "label2id": {
    "extreme": 2,
    "high": 4,
    "moderate": 0,
    "neutral": 3,
    "strong": 1
  },
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 514,
  "model_type": "xlm-roberta",
  "num_attention_heads": 16,
  "num_hidden_layers": 24,
  "output_past": true,
  "pad_token_id": 1,
  "position_embedding_type": "absolute",
  "

Epoch,Training Loss,Validation Loss,Accuracy,F1 Score,Precision,Recall,Roc Auc
1,1.379000,1.309856,0.433584,0.261498,0.268607,0.286704,nan
2,1.274700,1.252893,0.433584,0.266311,0.260491,0.288120,nan
3,1.180500,1.277136,0.431078,0.270664,0.272367,0.286253,nan
4,1.089200,1.308320,0.438596,0.274137,0.283386,0.291902,nan
5,1.002600,1.282254,0.456140,0.302242,0.374266,0.310056,nan
6,0.928600,1.331063,0.481203,0.315259,0.385652,0.321988,nan
7,0.835200,1.420358,0.473684,0.315012,0.354521,0.319686,nan
8,0.771300,1.561566,0.456140,0.330172,0.374587,0.324891,nan
9,0.697700,1.566969,0.473684,0.325458,0.352591,0.329073,nan
10,0.625300,1.748435,0.466165,0.332804,0.365076,0.332995,nan



***** Running Evaluation *****
  Num examples = 399
  Batch size = 32
NaN or Inf found in input tensor.
Saving model checkpoint to outputs/emotion-classifier-e5-2/checkpoint-59
loading configuration file config.json from cache at /home/paudan/.cache/huggingface/hub/models--intfloat--multilingual-e5-large/snapshots/3d7cfbdacd47fdda877c5cd8a79fbcc4f2a574f3/config.json
Model config XLMRobertaConfig {
  "architectures": [
    "XLMRobertaModel"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "dtype": "float32",
  "eos_token_id": 2,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 1024,
  "initializer_range": 0.02,
  "intermediate_size": 4096,
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 514,
  "model_type": "xlm-roberta",
  "num_attention_heads": 16,
  "num_hidden_layers": 24,
  "output_past": true,
  "pad_token_id": 1,
  "position_embedding_type": "absolute",
  "transformers_version": "4.56.0",
  "type_vocab

/opt/conda/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/conda/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/conda/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


              precision    recall  f1-score   support

    moderate       0.33      0.32      0.32       119
      strong       0.00      0.00      0.00        13
     extreme       0.17      0.05      0.07        21
     neutral       0.50      0.52      0.51       115
        high       0.49      0.58      0.53       132

    accuracy                           0.44       400
   macro avg       0.30      0.29      0.29       400
weighted avg       0.41      0.44      0.42       400

